In [ ]:
import string
import re

import pandas as pd
import numpy as np
import nltk


nltk.download('stopwords')
STOPWORDS = nltk.corpus.stopwords.words("english") + list(string.punctuation)

data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [ ]:
data.head()

In [ ]:
test_data.head()

In [ ]:
ids = test_data['id']

In [ ]:
# drop columns
to_drop = ['id', 'keyword', 'location']
data = data.drop(columns=to_drop)
test_data = test_data.drop(columns=to_drop)

In [ ]:
# all data
all_data = pd.concat([data, test_data])

In [ ]:
print(all_data.shape)

In [ ]:
# Metadata features
all_data['word_count'] = all_data['text'].apply(lambda x: len(x.split()))
all_data['char_count'] = all_data['text'].apply(len)

def avg_word_len(text):
    words = text.split()
    return np.mean([len(word) for word in words])

all_data['avg_word_len'] = all_data['text'].apply(avg_word_len)
all_data['num_unique_words'] = all_data['text'].apply(lambda x: len(np.unique(x.lower().split())))
all_data['num_stopwords'] = all_data['text'].apply(lambda x: len([word for word in x.lower().split()
                                                                 if word in STOPWORDS]))
all_data['num_urls'] = all_data['text'].apply(lambda x: len([word for word in x.lower().split() 
                                                             if 'http' in word or 'https' in word]))
all_data['stopword_ratio'] = all_data['num_stopwords'] / all_data['word_count']
all_data['punct_count'] = all_data['text'].apply(lambda x: len([word for word in x.lower().split()
                                                               if word in string.punctuation]))

In [ ]:
all_data.head()

In [ ]:
import autocorrect

LEMMATIZER = nltk.stem.WordNetLemmatizer()
# SPELLER = autocorrect.Speller(lang="en")

def preprocess(text):
    text = text.strip()
    text = text.lower()
#     text = " ".join([SPELLER(word) for word in text.split()])
    text = " ".join([word for word in text.split() if len(word) > 3])
    text = " ".join([word for word in text.split() if word.isalpha() == True])
    text = " ".join([LEMMATIZER.lemmatize(word) for word in text.split() if word not in STOPWORDS])
    text = re.sub(r"\d+", "", text)
    
    emoji_pattern = re.compile(pattern = "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags = re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    return text
    
all_data['text'] = all_data['text'].apply(preprocess)

In [ ]:
all_data['text'].head()

In [ ]:
all_texts = all_data['text'].to_numpy()

train = all_data[:len(data)].drop(columns='target')
labels = data['target']

test = all_data[len(data):].drop(columns='target')

In [ ]:
train.head()

In [ ]:
labels.head()

In [ ]:
test.head()

In [ ]:
print(all_texts.shape)

In [ ]:
from tensorflow import keras

NUM_WORDS = 1000
tokenizer = keras.preprocessing.text.Tokenizer(num_words=NUM_WORDS)
tokenizer.fit_on_texts(all_texts)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SCALER = StandardScaler()
X_train, X_val, y_train, y_val = train_test_split(train, labels, stratify=labels, 
                                                    test_size=0.2, random_state=42)

X_train_text = X_train['text'].to_numpy()
X_train_meta = SCALER.fit_transform(X_train.drop(columns='text').to_numpy())
y_train = y_train.to_numpy()

X_val_text = X_val['text'].to_numpy()
X_val_meta = SCALER.transform(X_val.drop(columns='text').to_numpy())
y_val = y_val.to_numpy()

X_test_text = test['text'].to_numpy()
X_test_meta = SCALER.transform(test.drop(columns='text').to_numpy())

X_train_text_encoded = tokenizer.texts_to_sequences(X_train_text)
X_val_text_encoded = tokenizer.texts_to_sequences(X_val_text)
X_test_text_encoded = tokenizer.texts_to_sequences(X_test_text)

In [ ]:
MAX_LEN = 30  # these are tweets after all...
X_train_processed = keras.preprocessing.sequence.pad_sequences(X_train_text_encoded, maxlen=MAX_LEN)
X_val_processed = keras.preprocessing.sequence.pad_sequences(X_val_text_encoded, maxlen=MAX_LEN)
X_test_processed = keras.preprocessing.sequence.pad_sequences(X_test_text_encoded, maxlen=MAX_LEN)

In [ ]:
print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

In [ ]:
print(X_train_meta.shape)
print(X_val_meta.shape)
print(X_test_meta.shape)

In [ ]:
# EMBEDDING_SIZE = 64

# model = keras.models.Sequential([
#     keras.layers.Embedding(NUM_WORDS, EMBEDDING_SIZE, input_length=MAX_LEN),
#     keras.layers.SpatialDropout1D(0.2),
# #     keras.layers.Conv1D(64, 3, padding='causal', activation='relu'),
# #     keras.layers.BatchNormalization(),
# #     keras.layers.Conv1D(96, 3, padding='causal', activation='relu'),
# #     keras.layers.BatchNormalization(),
# #     keras.layers.Conv1D(128, 3, padding='causal', activation='relu'),
# #     keras.layers.BatchNormalization(),
#     keras.layers.Bidirectional(keras.layers.LSTM(100, dropout=0.2, recurrent_dropout=0.2, 
#                                                  kernel_regularizer=keras.regularizers.l2(1e-4), 
#                                                  kernel_initializer='orthogonal')),
#     keras.layers.Dense(100, activation='relu'),
#     keras.layers.Dropout(0.2),
#     keras.layers.Dense(1, activation='sigmoid', kernel_regularizer=keras.regularizers.l2(1e-4))
# ])

In [ ]:
EMBEDDING_SIZE = 200

meta_input = keras.layers.Input(shape=(X_train_meta.shape[1],))
text_input = keras.layers.Input(shape=(MAX_LEN,))
embedding = keras.layers.Embedding(NUM_WORDS, EMBEDDING_SIZE, input_length=MAX_LEN)(text_input)
spatial_dropout = keras.layers.SpatialDropout1D(0.2)(embedding)
bidirectional_lstm = keras.layers.Bidirectional(
    keras.layers.LSTM(100, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=keras.regularizers.l2(1e-4),
                                 kernel_initializer = 'orthogonal'))(spatial_dropout)
concat = keras.layers.Concatenate()([bidirectional_lstm, meta_input])
dense = keras.layers.Dense(100, activation='relu')(concat)
dropout = keras.layers.Dropout(0.2)(dense)
output = keras.layers.Dense(1, activation='sigmoid', kernel_regularizer=keras.regularizers.l2(1e-4))(dropout)

model = keras.models.Model(inputs=[text_input, meta_input], outputs=output)

In [ ]:
EPOCHS = 50
BS = 64
LR = 3e-4
DECAY = LR // EPOCHS
PATIENCE = 4

optimizer = keras.optimizers.Adam(learning_rate=LR, decay=DECAY)
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE)

model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(X_train_processed, y_train, validation_data=(X_val_processed, y_val), 
#           batch_size=BS, epochs=EPOCHS, callbacks=[early_stop])

In [ ]:
model.fit([X_train_processed, X_train_meta], y_train, validation_data=([X_val_processed, X_val_meta], y_val), 
          batch_size=BS, epochs=EPOCHS, callbacks=[early_stop])

In [ ]:
import tensorflow as tf

final_predictions = tf.squeeze((model.predict([X_test_processed, X_test_meta]) > 0.5).astype(int))
# final_predictions = tf.squeeze(model.predict_classes(X_test_processed).astype(int))

In [ ]:
submission = pd.DataFrame({
    'id': ids,
    'target': final_predictions
})

In [ ]:
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)